# Fine-tuned Baselines: BERT, HateBERT, and RoBERTa

Fine-tune three pretrained models (BERT, HateBERT, RoBERTa) on three hate speech datasets (IHC, ISHate, Vicomtech) using standard binary cross-entropy training.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import warnings
warnings.filterwarnings("ignore")

# add src/ to path so shared modules (retriever, data_loaders, ...) are importable
sys.path.insert(0, str(Path("..").resolve()))

## 1. Load the Datasets

Load IHC, ISHate and Vicomtech using `data_loaders.py`.

In [ ]:
from data_loaders import load_ihc_binary, load_ishate_binary, load_vicomtech

# Load the 3 datasets using the functions in dataloaders.py and print the train/test lengths
train_ds, test_ds       = load_ihc_binary(seed=42)
ishate_train, ishate_test = load_ishate_binary()
vicomtech_train, vicomtech_test = load_vicomtech()

print("IHC")
print(f"  Train: {len(train_ds):,}  Test: {len(test_ds):,}")
print("\nISHate")
print(f"  Train: {len(ishate_train):,}  Test: {len(ishate_test):,}")
print("\n Vicomtech")
print(f"  Train: {len(vicomtech_train):,}  Test: {len(vicomtech_test):,}")

## 2. Tokenization

Tokenize each dataset using the model's own tokenizer, truncating and padding to 128 tokens.

In [3]:
def tokenize(ds, tokenizer, text_col="post", max_length=128):
    def _tok(batch):
        return tokenizer(
            batch[text_col],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
    return ds.map(_tok, batched=True)

## 3. Metrics

We use macro F1 as the primary metric, with precision and recall as additional diagnostic values.

In [ ]:
from training_utils import compute_metrics

## 4. Fine-tuning Loop

For each (model, dataset) pair: load the pretrained weights, fine-tune for 3 epochs, save the weights, and print the classification report.

In [ ]:
# Define a dictionnary of the baseline models "short name" and "real name"
MODELS = {
    "bert":     "bert-base-uncased",
    "hatebert": "GroNLP/hateBERT",
    "roberta":  "roberta-base",
}

# Create a dictionnary containing each datasets train/test data 
DATASETS = {
    "IHC":       {"train": train_ds,        "test": test_ds,        "text_col": "post"},
    "ISHate":    {"train": ishate_train,     "test": ishate_test,    "text_col": "text"},
    "Vicomtech": {"train": vicomtech_train,  "test": vicomtech_test, "text_col": "text"},
}

results = {}

# Go through every dataset
for dataset_name, dataset in DATASETS.items():
    results[dataset_name] = {}
    # Go through each model
    for model_name, model_id in MODELS.items():
        print(f"Dataset: {dataset_name}  |  Model: {model_name}")

        # Load model's tokenizer and model (with appropriate classification head added using AutoModelForSequenceClassification function)
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model     = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

        # Tokenize train/test data
        tok_train = tokenize(dataset["train"], tokenizer, text_col=dataset["text_col"])
        tok_test  = tokenize(dataset["test"],  tokenizer, text_col=dataset["text_col"])

        # Define the training hyperparameters and strategies
        training_args = TrainingArguments(
            output_dir=f"../../checkpoints_baseline/{dataset_name}/{model_name}",
            num_train_epochs=3,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            learning_rate=2e-5,
            eval_strategy="epoch",
            save_strategy="no",
            logging_strategy="epoch",
            report_to="none",
            seed=42,
        )

        # Define the trainer
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tok_train,
            eval_dataset=tok_test,
            compute_metrics=compute_metrics,
        )

        # Train
        trainer.train()

        # Save fine-tuned weights in HuggingFace format
        save_path = f"../../weigths/weights_baseline/{model_name}/{dataset_name}"
        os.makedirs(save_path, exist_ok=True)
        trainer.save_model(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"Saved weights → {save_path}")

        # Test the model
        preds_output = trainer.predict(tok_test)
        preds  = np.argmax(preds_output.predictions, axis=-1)
        labels = dataset["test"]["label"]

        # Prints its performances
        print(classification_report(labels, preds, target_names=["Non-HS", "HS"]))

        # Add them in the results table
        results[dataset_name][model_name] = {
            "macro_f1":  f1_score(labels, preds, average="macro",  zero_division=0),
            "macro_p":   precision_score(labels, preds, average="macro", zero_division=0),
            "macro_r":   recall_score(labels, preds, average="macro",    zero_division=0),
        }

## 5. Results

One table per dataset showing macro F1, precision and recall for all three models.

In [ ]:
import pandas as pd

# Display the results in a table using panda's functions
for dataset_name, dataset_results in results.items():
    df = pd.DataFrame(dataset_results).T
    df.columns = ["Macro F1", "Macro Precision", "Macro Recall"]
    df.index.name = "Model"
    styled = df.style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold; background-color: #d4f1d4").set_caption(dataset_name)
    display(styled)

ValueError: Length mismatch: Expected axis has 0 elements, new values have 3 elements